In [1]:
a = int(input("enter a number: "))
b = int(input("enter another number: "))

print("Two relational numbers is: ")
print("Value of a =", a)
print("Value of b =", b)

print("a > b  =", a > b)
print("a < b  =", a < b)
print("a >= b =", a >= b)
print("a <= b =", a <= b)
print("a == b =", a == b)
print("a != b =", a != b)

Two relational numbers is: 
Value of a = 2
Value of b = 3
a > b  = False
a < b  = True
a >= b = False
a <= b = True
a == b = False
a != b = True


In [2]:
a = int(input("Enter a number: "))
b = int(input("Enter another number: "))
c = int(input("Enter third number: "))

print("\nValues:")
print("a =", a, "b =", b, "c =", c)

print("\nOperator Precedence Results:")

# Parentheses ()
result1 = (a + b) * c
print("(a + b) * c =", result1)

# Exponentiation **
result2 = a ** c
print("a ** c =", result2)

# Unary operators + -
result3 = -a + b
print("-a + b =", result3)

# Multiplication, Division, Modulus, Floor Division
result4 = a * b / c
print("a * b / c =", result4)

result5 = a // c
print("a // c =", result5)

result6 = a % c
print("a % c =", result6)

# Addition and Subtraction
result7 = a + b - c
print("a + b - c =", result7)


Values:
a = 4 b = 5 c = 5

Operator Precedence Results:
(a + b) * c = 45
a ** c = 1024
-a + b = 1
a * b / c = 4.0
a // c = 0
a % c = 4
a + b - c = 4


In [3]:
import ast
import operator

class PrecedenceSolver:
    def __init__(self):
        self.variables = {}
        self.op_table = {
            ast.Pow:      ('**', 'Exponentiation',  1, operator.pow),
            ast.Mult:     ('*',  'Multiplication',   2, operator.mul),
            ast.Div:      ('/',  'Division',         2, operator.truediv),
            ast.FloorDiv: ('//', 'Floor Division',   2, operator.floordiv),
            ast.Mod:      ('%',  'Modulus',          2, operator.mod),
            ast.Add:      ('+',  'Addition',         3, operator.add),
            ast.Sub:      ('-',  'Subtraction',      3, operator.sub),
        }
        
    def set_variables(self, vars_dict):
        self.variables = vars_dict
    
    def parse_expression(self, expr):
        """Expression ko clean karke parse karta hai"""
        return ast.parse(expr, mode='eval').body
    
    def analyze_operators(self, expr):
        """Saare operators detect karta hai"""
        found = []
        def walk(node):
            if isinstance(node, ast.BinOp):
                sym, name, prec, _ = self.op_table[type(node.op)]
                found.append((sym, name, prec))
                walk(node.left)
                walk(node.right)
            elif isinstance(node, ast.UnaryOp) and isinstance(node.op, ast.USub):
                found.append(('-', 'Unary Minus', 0.5))
                walk(node.operand)
        
        walk(self.parse_expression(expr))
        return list(set(found))  # unique operators
    
    def evaluate_step_by_step(self, expr):
        """Step by step evaluation with detailed breakdown"""
        tree = self.parse_expression(expr)
        steps = []
        step_num = [0]
        
        def eval_node(node):
            if isinstance(node, ast.Constant):
                return node.value
            elif isinstance(node, ast.Name):
                return self.variables[node.id]
            elif isinstance(node, ast.BinOp):
                left_val  = eval_node(node.left)
                right_val = eval_node(node.right)
                sym, name, prec, func = self.op_table[type(node.op)]
                
                result = func(left_val, right_val)
                step_num[0] += 1
                
                # Clean output
                if isinstance(result, float) and result.is_integer():
                    result = int(result)
                
                step_info = {
                    'step': step_num[0],
                    'op': sym,
                    'name': name,
                    'left': left_val,
                    'right': right_val,
                    'result': result,
                    'priority': prec
                }
                steps.append(step_info)
                return result
            elif isinstance(node, ast.UnaryOp) and isinstance(node.op, ast.USub):
                val = eval_node(node.operand)
                step_num[0] += 1
                steps.append({
                    'step': step_num[0],
                    'op': '-',
                    'name': 'Unary Minus',
                    'left': None,
                    'right': val,
                    'result': -val,
                    'priority': 0.5
                })
                return -val
        
        final_result = eval_node(tree)
        return steps, final_result
    
    def pretty_print_analysis(self, expr):
        """Complete analysis print karta hai"""
        print(f"\n{'='*70}")
        print(f"📊  EXPRESSION ANALYSIS:  {expr}")
        print(f"{'='*70}")
        
        # 1. Detect operators
        operators = self.analyze_operators(expr)
        print(f"\n🔍  DETECTED OPERATORS:")
        print(f"   {'Priority':<8} {'Symbol':<4} {'Name'}")
        print(f"   {'-'*35}")
        for _, (sym, name, prec) in sorted(operators, key=lambda x: x[1][2]):
            print(f"   {prec:<8} {sym:<4} {name}")
        
        # 2. Step by step solution
        steps, result = self.evaluate_step_by_step(expr)
        
        print(f"\n⚡  STEP-BY-STEP SOLUTION (High Priority PEHLE):")
        print(f"   {'Step':<4} {'Priority':<8} {'Operation'}")
        print(f"   {'-'*50}")
        
        for step in steps:
            left_str = str(step['left'])
            right_str = str(step['right'])
            result_str = str(step['result'])
            
            operation = f"{left_str} {step['op']} {right_str}"
            print(f"   {step['step']:<4} {step['priority']:<8} {operation} = {result_str:<15}  ← {step['name']}")
        
        print(f"\n🎯  FINAL RESULT:  {expr}  =  {result}")
        print(f"{'='*70}")

# ===========================================
#               MAIN PROGRAM
# ===========================================

solver = PrecedenceSolver()

# Input
print("🔢  Enter Values:")
a = int(input("  a = "))
b = int(input("  b = "))
c = int(input("  c = "))

solver.set_variables({'a': a, 'b': b, 'c': c})

print(f"\n✅  Values set: a={a}, b={b}, c={c}")
print("\n📈  Testing Common Expressions:\n")

# Test cases
test_expressions = [
    "a + b * c",
    "(a + b) * c",
    "a ** c + b",
    "a * b / c",
    "a + b - c // 2",
    "-a + b * c % 3",
]

for expr in test_expressions:
    solver.pretty_print_analysis(expr)
    input("\n⏸️  [Press Enter for next...]")

# Custom input
print("\n🎮  CUSTOM MODE:")
print("   Use a, b, c with operators: + - * / // % ** ()")
while True:
    user_expr = input("\n📝  Enter expression (or 'quit'): ").strip()
    if user_expr.lower() in ['quit', 'q', 'exit']:
        print("\n👋  Thanks for using Operator Precedence Solver!")
        break
    try:
        solver.pretty_print_analysis(user_expr)
    except Exception as e:
        print(f"❌  Error: {e}")
        print("   Try again with valid expression!")

🔢  Enter Values:

✅  Values set: a=6, b=6, c=66

📈  Testing Common Expressions:


📊  EXPRESSION ANALYSIS:  a + b * c

🔍  DETECTED OPERATORS:
   Priority Symbol Name
   -----------------------------------


ValueError: too many values to unpack (expected 2)